In [1]:
!pip install -U langchain
!pip install -U langchain-cohere
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-chroma
!pip install -U pypdf
!pip install -U chromadb

  Using cached langchain_cohere-0.6.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached cohere-5.21.1-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 9.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.7 MB/s eta 0:00:00a 0:00:01
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_classic-1.0.8-py3-none-any.whl (1.0 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: requests
    Found existing installation:

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("coherekey")

----

## **Part 1 — Test the RAG with Your Own CV**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/kaggle/input/datasets/islamohamed10/my-cv-v2/CV.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

Number of pages: 2
Islam Mohamed
♂¶ap-¶arker-altCairo, Egypt✉10islammohamed01@gmail.com♂phone-alt+20 114 468 2583/linkedin-inIslam Mohamed
/githubislam0114
Summary
Computer and Communication Engineering student at Shoubra Engineering with a strong pas-
sion for Artificial Intelligence and Machine Learning. Actively building skills in machine learning,
deep learning, data preprocessing, and data analysis through intensive training programs and cer-
tifications from IBM, Microsoft, NTI, ITI, DEPI, and DataCamp. Know


In [6]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_rag"
)

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [10]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
Islam Mohamed
♂¶ap-¶arker-altCairo, Egypt✉10islammohamed01@gmail.com♂phone-alt+20 114 468 2583/linkedin-inIslam Mohamed
/githubislam0114
Summary
Computer and Communication Engineering student at Shoubra Engineering with a strong pas-
sion for Artificial Intelligence and Machine Learning. Actively building skills in machine learning,
deep learning, data preprocessing, and data analysis through intensive training programs and cer-
tifications from IBM, Microsoft, NTI, ITI, DEPI, and DataCamp. Known for being organized,
disciplined with time, and a critical thinker who enjoys learning and exploring new AI concepts.
Currently seeking internship or junior roles to gain practical experience, apply theoretical knowl-
edge, and grow into a distinguished professional in the AI field.
Education
Metadata: {'page': 0, 'title': "Islam Mohamed's CV", 'producer': 'pdfTeX-1.40.27', 'creationdate': '2026-06-12T20:06:07+00:00', 'author': 'Islam Mohamed', 'source': '/kaggle/input/data

In [11]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [13]:
from langchain_core.runnables import RunnablePassthrough

In [14]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [15]:
question = "•	What are my main technical skills?"

response = rag_chain.invoke(question)

print(response.content)

Your main technical skills include:

- **Programming**: Python, Pandas, NumPy, Matplotlib, Seaborn, SciKit-learn, TensorFlow, Keras, PyTorch  
- **Machine Learning**: Supervised and Unsupervised Learning Algorithms, Model Training and Evaluation, Feature Selection, Hyperparameter Tuning  
- **Deep Learning**: Neural Networks, CNN, RNN, Transformers  
- **Data Science**: Data Preprocessing, EDA, Statistical Analysis, Data Visualization  
- **Deployment Tools**: Git, GitHub, Streamlit


In [16]:
question = "•	What machine learning experience do I have?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, your machine learning experience includes:

1. **Supervised and Unsupervised Learning**: Developed and fine-tuned models using algorithms like Random Forest, with a focus on feature selection and hyperparameter tuning to improve performance.  
2. **Deep Learning**: Explored architectures such as CNN (Convolutional Neural Networks) and RNN (Recurrent Neural Networks), and gained foundational experience in Transformers.  
3. **Model Training and Evaluation**: Performed data preprocessing, feature selection, and model fine-tuning to ensure high accuracy and explainability of predictions.  
4. **Projects**:  
   - Built a Random Forest model for **Bank Customer Churn Prediction**, achieving robust performance on imbalanced datasets.  
   - Developed a **Bilingual Semantic Chatbot** using a RAG pipeline with Sentence-BERT and Google Gemini API.  
5. **Tools and Frameworks**: Utilized Python, Scikit-learn, TensorFlow, Keras, PyTorch, and Streamlit for machine l

In [17]:
question = "•	What projects are mentioned in my CV?"

response = rag_chain.invoke(question)

print(response.content)

The projects mentioned in your CV are:

1. **Moshrif – Smart Attendance & University ERP System**: Built the core AI vision engine for a 4-layer microservices ERP system to fully automate university attendance.  
2. **Book Recommendation System**: Utilized 35MB ratings datasets to deliver personalized book suggestions based on user behavior.  
3. **Bilingual Semantic Chatbot**: Built a RAG pipeline using paraphrase-multilingual-MiniLM-L12-v2 and Google Gemini API.  
4. **Bank Customer Churn Project**: Developed and deployed a Random Forest model to predict bank customers at risk of churn, with an interactive Streamlit and Power BI dashboard for real-time insights.


In [18]:
question = "•	What programming languages do I know?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, the programming languages you know include:

- **Python** (extensively used across various projects and experiences)
- **React.js** (mentioned in the "Data Integration" project)

No other programming languages are explicitly listed in the context.


In [19]:
question = "•	What training or internships have I completed?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, you have completed the following training and internships:

1. **Data Science and Machine Learning Trainee, DEPI** (Dec 2025 – Jul 2026)  
2. **AI Summer Training – ITI** (Aug 2025)  
3. **Machine Learning – NTI** (Aug 2025)  
4. **AI & Machine Learning – Sprints x Microsoft** (Oct 2025)  
5. **IBM AI Engineer Certificate – Coursera** (Jan 2026)  
6. **AI & Data Science – DEPI** (May 2025)  

Additionally, you are currently seeking internship or junior roles to gain further practical experience.


----